# Regression Analysis

This notebook performs a multiple linear regression to predict the number of road accidents
in Italian municipalities based on population, surface area and year.
The analysis includes multicollinearity checks, model refinement and performance evaluation
using R², RMSE and MAPE metrics.

In [1]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import sys
sys.path.append('../config')
from config import *

In [2]:
# import clustered_istat_data from clean
df_merged_clustered_istat_data=pd.read_csv(CLEAN_PATH + 'clustered_istat_data.csv')

# display df_merged_clustered_istat_data
df_merged_clustered_istat_data.head()

,REF_AREA,TIME_PERIOD,KILLINJ,ROADACC,Comune,Superficie (Kmq),Popolazione residente,Anno,ROADACC_PER_CAPITA,ROADACC_PER_KM2,ROADACC_PER_CAPITA_scaled,ROADACC_PER_KM2_scaled,cluster,cluster_label
0,1001,2001,10,5,Agliè,13.1462,2557.0,2001.0,0.001955,0.380338,-0.015349,-0.161350,0,Low Risk
1,1001,2002,10,5,Agliè,13.1462,2538.0,2002.0,0.001970,0.380338,-0.009130,-0.161350,0,Low Risk
2,1001,2003,7,4,Agliè,13.1462,2588.0,2003.0,0.001546,0.304270,-0.189448,-0.198299,0,Low Risk
3,1001,2004,13,9,Agliè,13.1462,2679.0,2004.0,0.003359,0.684608,0.581114,-0.013555,1,High Per Capita
4,1001,2005,2,2,Agliè,13.1462,2674.0,2005.0,0.000748,0.152135,-0.528304,-0.272197,0,Low Risk


In [3]:
# create x predictor and y target variables
x_predictors=df_merged_clustered_istat_data[['Popolazione residente','Superficie (Kmq)', 'TIME_PERIOD', 'KILLINJ']]
y_target=df_merged_clustered_istat_data['ROADACC']

# split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(x_predictors, y_target, test_size=0.2, random_state=42)

# create a linear regression model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(y_pred)


[-0.55929345 -0.91289004 33.86375904 ... 19.40094539  0.14557362
 -2.48474127]


## Multicollinearity Check

The model predicted negative values for some communes, which is not realistic for accident counts.
Additionally, the R² score of 0.996 is suspiciously high, suggesting that one of the predictors 
may be too correlated with the target variable ROADACC.
We check the correlation matrix to identify any multicollinearity issues before refining the model.

In [4]:
# calculate the R-squared value for the model
r_squared = r2_score(y_test, y_pred)

# calculate the RMSE (Root Mean Squared Error)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# display r_squared and rmse
print("R-squared:", r_squared)
print("RMSE:", rmse)

# check for multicollinearity using VIF
corr_matrix = df_merged_clustered_istat_data[['KILLINJ', 'ROADACC', 'Popolazione residente', 'Superficie (Kmq)', 'TIME_PERIOD']].corr()

# display the correlation matrix
print(corr_matrix)



R-squared: 0.9959831109334251
RMSE: 11.846017433157666
                        KILLINJ   ROADACC  Popolazione residente  \
KILLINJ                1.000000  0.998535               0.921893   
ROADACC                0.998535  1.000000               0.919423   
Popolazione residente  0.921893  0.919423               1.000000   
Superficie (Kmq)       0.334228  0.326003               0.383531   
TIME_PERIOD           -0.017530 -0.014897               0.005619   

                       Superficie (Kmq)  TIME_PERIOD  
KILLINJ                        0.334228    -0.017530  
ROADACC                        0.326003    -0.014897  
Popolazione residente          0.383531     0.005619  
Superficie (Kmq)               1.000000     0.010508  
TIME_PERIOD                    0.010508     1.000000  


## Removing KILLINJ from Predictors

The correlation matrix shows a correlation of 0.998 between KILLINJ and ROADACC, which is almost perfect.
This means KILLINJ is essentially the same variable as ROADACC, causing multicollinearity.
Including it in the model gives an artificially high R² score without adding real predictive value.
We therefore remove KILLINJ from the predictors and retrain the model.

In [5]:
# create x predictor and y target variables

x_predictors=df_merged_clustered_istat_data[['Popolazione residente','Superficie (Kmq)', 'TIME_PERIOD']]
y_target=df_merged_clustered_istat_data['ROADACC']

# split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(x_predictors, y_target, test_size=0.2, random_state=42)

# create a linear regression model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(y_pred)



[-12.8687325  -11.88592143  34.00682368 ...  34.50143348 -16.26659301
 -31.3155746 ]


In [6]:
# calculate the R-squared value for the model
r_squared = r2_score(y_test, y_pred)

# calculate the RMSE (Root Mean Squared Error)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# display r_squared and rmse
print("R-squared:", r_squared)
print("RMSE:", rmse)

R-squared: 0.8244175602429216
RMSE: 78.31915408478278


In [7]:
# calculate MAPE excluding rows where y_test is 0
mask = y_test != 0
mape = np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100
print("MAPE:", mape)

MAPE: 382.6290954759531


## Model Results

After removing KILLINJ from the predictors, the model shows more realistic results.
R² of 0.824 means the model explains 82% of the variance in road accidents, which is a good result.
RMSE of 78.3 means the model predicts the number of accidents with an average error of 78 accidents per year.
However, the MAPE of 382% indicates that the model performs poorly on small communes with very few accidents,
where even a small absolute error results in a large percentage error.
The model is more reliable for larger cities with high accident counts.
Some negative predictions persist for small communes, as linear regression has no lower bound constraint.

## Adding Cluster as Predictor

To try to improve the model, we add the cluster label as a predictor.
Since cluster_label is a categorical variable, we need to convert it into dummy variables
using pd.get_dummies() before adding it to the model.
We use drop_first=True to avoid the dummy variable trap, which would cause multicollinearity.

In [8]:
# convert cluster_label with pd.get_dummies()
dummies = pd.get_dummies(df_merged_clustered_istat_data['cluster_label'], drop_first=True)

# concat dummies into df_merged_clustered_istat_data
df_merged_clustered_istat_data = pd.concat([df_merged_clustered_istat_data, dummies], axis=1)

# display the first 5 rows of the updated DataFrame
df_merged_clustered_istat_data.head()


,REF_AREA,TIME_PERIOD,KILLINJ,ROADACC,Comune,Superficie (Kmq),Popolazione residente,Anno,ROADACC_PER_CAPITA,ROADACC_PER_KM2,ROADACC_PER_CAPITA_scaled,ROADACC_PER_KM2_scaled,cluster,cluster_label,High Per Km2,Low Risk
0,1001,2001,10,5,Agliè,13.1462,2557.0,2001.0,0.001955,0.380338,-0.015349,-0.161350,0,Low Risk,False,True
1,1001,2002,10,5,Agliè,13.1462,2538.0,2002.0,0.001970,0.380338,-0.009130,-0.161350,0,Low Risk,False,True
2,1001,2003,7,4,Agliè,13.1462,2588.0,2003.0,0.001546,0.304270,-0.189448,-0.198299,0,Low Risk,False,True
3,1001,2004,13,9,Agliè,13.1462,2679.0,2004.0,0.003359,0.684608,0.581114,-0.013555,1,High Per Capita,False,False
4,1001,2005,2,2,Agliè,13.1462,2674.0,2005.0,0.000748,0.152135,-0.528304,-0.272197,0,Low Risk,False,True


In [9]:
# create x predictor and y target variables

x_predictors=df_merged_clustered_istat_data[['Popolazione residente','Superficie (Kmq)', 'TIME_PERIOD', 'High Per Km2', 'Low Risk']]
y_target=df_merged_clustered_istat_data['ROADACC']

# split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(x_predictors, y_target, test_size=0.2, random_state=42)

# create a linear regression model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(y_pred)

[-12.99476602 -12.03171666  35.05402615 ...  34.393175   -16.3603996
 -31.51071794]


In [10]:
# calculate the R-squared value for the model
r_squared = r2_score(y_test, y_pred)

# calculate the RMSE (Root Mean Squared Error)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# display r_squared and rmse
print("R-squared:", r_squared)
print("RMSE:", rmse)

R-squared: 0.824376595235761
RMSE: 78.32828984346126


In [11]:
# calculate MAPE excluding rows where y_test is 0
mask = y_test != 0
mape = np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100
print("MAPE:", mape)

MAPE: 382.1623941117557


## Model Comparison

Adding the cluster dummies did not improve the model performance.
R² remains at 0.824 and RMSE at 78.3, identical to the previous model.
This suggests that the cluster information is already captured by the existing predictors
(Popolazione residente, Superficie (Kmq), TIME_PERIOD), and does not add new information to the model.
The high MAPE of 382% confirms that the model struggles with small communes that have very few accidents,
which is a known limitation of linear regression on skewed datasets.

## Filtering Small Municipalities

To address the high MAPE caused by small municipalities with very few accidents,
we try filtering out municipalities below a certain accident threshold.
This allows the model to focus on municipalities where predictions are more meaningful
and reduces the impact of outliers with near-zero accident counts.

In [12]:
# check statistics for roadacc
df_merged_clustered_istat_data['ROADACC'].describe()

count    190900.000000
mean         25.002378
std         257.699829
min           0.000000
25%           1.000000
50%           3.000000
75%          12.000000
max       23135.000000
Name: ROADACC, dtype: float64

In [13]:
# filter municipalities with more than 12 accidents
df_filtered = df_merged_clustered_istat_data[df_merged_clustered_istat_data['ROADACC'] > 12]

In [14]:
# create x predictor and y target variables

x_predictors=df_filtered[['Popolazione residente','Superficie (Kmq)', 'TIME_PERIOD']]
y_target=df_filtered['ROADACC']

# split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(x_predictors, y_target, test_size=0.2, random_state=42)

# create a linear regression model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(y_pred)

[ -0.54664799  33.36984113  36.22431838 ...  11.23992012 -18.29583886
 159.58632671]


In [15]:
# calculate the R-squared value for the model
r_squared = r2_score(y_test, y_pred)

# calculate the RMSE (Root Mean Squared Error)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# display r_squared and rmse
print("R-squared:", r_squared)
print("RMSE:", rmse)

R-squared: 0.8611477996479027
RMSE: 180.97901940011195


In [16]:
# calculate MAPE excluding rows where y_test is 0
mask = y_test != 0
mape = np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100
print("MAPE:", mape)

MAPE: 100.14901844363379


## Filtered Model Results

After filtering communes with less than 12 accidents per year (75th percentile),
the model performance improved significantly:

- R² improved from 0.824 to 0.861, meaning the model now explains 86% of the variance
- MAPE improved from 382% to 100%, a significant reduction in relative error
- RMSE increased from 78 to 181, which is expected as we are now working with communes 
  that have higher accident counts

The filtered model is more reliable and meaningful for business decisions,
as it focuses on communes where accurate predictions matter most.
Small communes with fewer than 12 accidents per year are less relevant 
for a road traffic management company.

## Excluding COVID-19 Year (2020)

The year 2020 represents a structural outlier due to COVID-19 lockdowns, which drastically 
reduced traffic across Italy and distorted the normal accident patterns.
Including this year in the model training may negatively affect predictions for normal years.
We therefore create a new filtered dataset excluding 2020 and retrain the model.

In [22]:
# create a copy of the original dataframe for filtering
df_filtered_covid = df_filtered

# exclude 2020 data as it is a structural outlier due to COVID-19 lockdowns
df_filtered_covid = df_filtered_covid[df_filtered_covid['TIME_PERIOD'] != 2020]

# create x predictor and y target variables
x_predictors=df_filtered_covid[['Popolazione residente','Superficie (Kmq)', 'TIME_PERIOD']]
y_target=df_filtered_covid['ROADACC']

# split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(x_predictors, y_target, test_size=0.2, random_state=42)

# create a linear regression model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(y_pred)


[ 192.68347307   94.52462291  370.08279598 ...   29.61925151   13.67107982
 1505.11673777]


In [23]:
# calculate the R-squared value for the model
r_squared = r2_score(y_test, y_pred)

# calculate the RMSE (Root Mean Squared Error)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# display r_squared and rmse
print("R-squared:", r_squared)
print("RMSE:", rmse)

# calculate MAPE excluding rows where y_test is 0
mask = y_test != 0
mape = np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100
print("MAPE:", mape)

R-squared: 0.8696670790456833
RMSE: 193.46147061838073
MAPE: 96.21846656012337


## Final Model Comparison

Excluding 2020 further improved the model performance:
- R² improved from 0.861 to 0.870, meaning the model now explains 87% of the variance
- MAPE improved from 100% to 96%, a small but meaningful reduction in relative error
- RMSE increased slightly from 181 to 193, expected as the training set is smaller

The final model uses communes with more than 12 accidents per year and excludes 2020,
providing the most reliable predictions for normal traffic conditions.
This model can be used to forecast accident counts for Italian municipalities
based on population, surface area and year.